# 📊 INF01090 - Ciência de Dados - Classification Techniques

**Spaceship Titanic - Kaggle-Style Competition**

Spaceship Titanic (<https://www.kaggle.com/competitions/spaceship-titanic/>) is a popular Kaggle competition that challenges participants to predict whether a passenger was transported to an alternate dimension, using classification models. It is one of Kaggle’s most popular “Getting Started” challenges, designed to build practical skills in data cleaning, feature engineering, and predictive modelling.

## Key facts

- **Platform:** Kaggle
- **Launch year:** 2016
- **Original dataset:** 8,693 training passengers
- **Features:** 13 explanatory variables
- **Primary goal:** Predict whether a passenger was transported


## 📂 Dataset and Objective

The dataset describes various attributes of passengers—covering their home planet, cryosleep status, cabin, age, VIP status, and amenities usage. The task is to predict the binary outcome of whether they were transported, making it a supervised classification problem.

The file **`data_description.txt`** has the full description of each column.

## 🛠 Skills and techniques

This competition is widely used to practise:

- **Feature engineering:** handling missing data, encoding categorical variables, transforming skewed features
- **Model building:** experimenting with algorithms such as logistic regression, random forest, and gradient boosting classifiers
- **Evaluation:** models are ranked by **Accuracy** in our local competition setup


## 🎯 Assignment Goal

Your goal is to build a classification pipeline that predicts **`Transported`** for the hidden test set.

This is not only a leaderboard exercise. You should use this assignment to demonstrate that you understand:

- data cleaning for tabular data
- feature encoding and transformation
- classification modeling
- error analysis
- the effect of different design choices on predictive performance


## 📁 Files You Will Receive

You should work only with the files distributed for this lab:

- **`train_student.csv`** — training data with the target column
- **`test_student.csv`** — test data without the target column
- **`submission_template.csv`** — expected format for submission
- **`data_description.txt`** — attribute descriptions

Do **not** use Kaggle's original public test split for submission. The grading app uses a **custom hidden split** created for this class.


## 🏁 Submission and Leaderboard

Submissions are evaluated in the local grading app:

<https://lab06inf01090-puo9j5ajrfsar22xxfmeez.streamlit.app/>

The app expects a CSV file with exactly these columns:

```csv
PassengerId,prediction
8141_01,0
6034_01,0
3946_01,1
```

Rules:

- `Id` must match the IDs in **`test_student.csv`**
- `prediction` must contain one numeric prediction per row
- all test rows must be present
- predictions for `Transported` should be True or False


## 📌 What You Must Deliver

Each group must submit:

1. **A prediction file** for the leaderboard  
2. **This notebook** (completed, with code, outputs, and short explanations)  
3. **A short report section in the notebook** explaining:
   - preprocessing choices
   - feature engineering
   - model(s) tested
   - final model used
   - interpretation of the obtained score


## 👥 Group Work

- Work in groups of up to 4
- All members of the group must understand the final solution
- Use a consistent team name in the leaderboard
- The same team may submit multiple times; the leaderboard keeps the best score


## 📊 Grading Criteria

Your grade will not depend only on leaderboard position. The first three places will get additional grade.

Important:
- a top leaderboard score with poor documentation is **not enough**
- a strong notebook with solid methodology can still receive a high grade even if it is not the top-ranked solution


## 🚦 Recommended Workflow

A good workflow for this assignment is:

1. Inspect the training data
2. Identify numeric and categorical variables
3. Handle missing values
4. Encode categorical variables
5. Optionally transform skewed variables
6. Build a baseline classification model
7. Evaluate improvements using validation on the training set
8. Train your final model on the full student training set
9. Predict on `test_student.csv`
10. Submit the predictions to the leaderboard


## ⚠️ Restrictions and Good Practice

- Do not manually inspect or reconstruct the hidden target values
- Do not hard-code predictions
- Do not submit malformed files to probe the scorer
- Do not use the leaderboard as your only validation method

Recommended:
- create your own validation split from `train_student.csv`
- compare models locally before submitting
- submit only meaningful improvements


## 🧪 Suggested Experiments

You may explore ideas such as:

- dropping columns with many missing values
- imputing missing values numerically and categorically
- one-hot encoding categorical variables
- checking class balance for Transported
- trying different regularization strengths
- comparing linear and non-linear models
- checking whether some features are highly skewed


## 🧭 Starter Checklist

Before your first submission, verify that:

- [x] `train_student.csv` loads correctly
- [x] `test_student.csv` has the same predictor columns as expected
- [x] your preprocessing works for both train and test
- [x] your model produces one prediction per test row
- [x] the output file has exactly two columns: `Id`, `prediction`
- [x] all predictions are numeric
- [ ] all predictions are boolean (True/False)


## 🐍 Suggested Notebook Structure

You may organize your work using sections such as:

1. Data loading
2. Exploratory inspection
3. Missing-value handling
4. Feature encoding
5. Train/validation split
6. Baseline model
7. Improved model
8. Final training and test prediction
9. Submission file generation
10. Reflection


In [11]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import optuna

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    cross_val_score
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

from sklearn.preprocessing import OrdinalEncoder

from sklearn.ensemble import (
    ExtraTreesClassifier,
    StackingClassifier
)

from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

## $\S\, 1.$ Carregamento dos Dados

In [12]:

# CONFIG
RANDOM_STATE = 42
N_SPLITS     = 3
N_TRIALS     = 5
TARGET = "Transported"
ID_COL = "PassengerId"

# LOAD
TRAIN_PATH = "data/processed/train_processed.csv"
TEST_PATH  = "data/processed/test_processed.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("\n==============================")
print("DATA LOADED")
print("==============================")

print("Train:", train_df.shape)
print("Test :", test_df.shape)



DATA LOADED
Train: (6085, 29)
Test : (2608, 29)


## 2. Identify target and predictors


In [13]:
test_ids = test_df[ID_COL].copy()
y = train_df[TARGET].astype(int)

X = train_df.drop(columns=[TARGET])
X = X.drop(columns=[ID_COL])

test_df = test_df.drop(columns=[ID_COL])

if TARGET in test_df.columns:
    test_df = test_df.drop(columns=[TARGET])
    
missing_in_test = set(X.columns) - set(test_df.columns)
missing_in_train = set(test_df.columns) - set(X.columns)

for col in missing_in_test:
    test_df[col] = np.nan

for col in missing_in_train:
    X[col] = np.nan

# Garantir mesma ordem
test_df = test_df[X.columns]

combined = pd.concat(
    [X, test_df],
    axis=0,
    ignore_index=True
)

print("\nDimensões combinadas dos dados processados:", combined.shape)

missing_total = combined.isnull().sum().sum()

print("\nValores ausentes:", missing_total)
if missing_total > 0:
    print(
        combined
        .isnull()
        .sum()
        .sort_values(ascending=False)
        .head(20)
    )
    
bool_cols = combined.select_dtypes(
    include=["bool"]
).columns

for col in bool_cols:

    combined[col] = combined[col].astype(int)    


Dimensões combinadas dos dados processados: (8693, 27)

Valores ausentes: 1160
HomePlanet       201
Family           200
Cabin_Deck       199
Cabin_Side       199
Destination      182
Age_Group        179
CryoSleep          0
NoExpenditure      0
Group_Size         0
Alone              0
VIP                0
Cabin_Region1      0
Cabin_Region2      0
Cabin_Region4      0
Cabin_Region3      0
Cabin_Region5      0
Cabin_Region6      0
Cabin_Region7      0
Family_Size        0
Age_yj             0
dtype: int64


## 4. Separate numeric and categorical columns


In [14]:
num_cols = combined.select_dtypes(
    include=[np.number]
).columns

for col in num_cols:
    combined[col] = combined[col].fillna(
        combined[col].median()
    )

cat_cols = combined.select_dtypes(
    include=["object", "category"]
).columns

for col in cat_cols:
    combined[col] = combined[col].fillna(
        "Unknown"
    )

if len(cat_cols) > 0:

    encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    combined[cat_cols] = encoder.fit_transform(
        combined[cat_cols].astype(str)
    )

final_missing = combined.isnull().sum().sum()
print("\nFinal Missing Values:", final_missing)
assert final_missing == 0, "Ainda existem missing values."

X = combined.iloc[:len(train_df)].copy()
test_df = combined.iloc[len(train_df):].copy()

print("\ndim. treino:", X.shape)
print("dim. teste:", test_df.shape)


Final Missing Values: 0

dim. treino: (6085, 27)
dim. teste: (2608, 27)


In [15]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

def objective_lgbm(trial):
    params = {
        "n_estimators": 300,
        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.01,
                0.1,
                log=True
            ),

        "max_depth":
            trial.suggest_int(
                "max_depth",
                4,
                10
            ),

        "num_leaves":
            trial.suggest_int(
                "num_leaves",
                16,
                128
            ),

        "subsample":
            trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

        "colsample_bytree":
            trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),

        "random_state": RANDOM_STATE,
        "verbosity": -1,
        "n_jobs": -1
    }

    model = LGBMClassifier(**params)

    scores = cross_val_score(
        model,
        X,
        y,
        cv=3,
        scoring="accuracy",
        n_jobs=-1
    )

    return scores.mean()

def objective_xgb(trial):
    params = {
        "n_estimators": 300,
        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.01,
                0.1,
                log=True
            ),

        "max_depth":
            trial.suggest_int(
                "max_depth",
                3,
                8
            ),

        "subsample":
            trial.suggest_float(
                "subsample",
                0.7,
                1.0
            ),

        "colsample_bytree":
            trial.suggest_float(
                "colsample_bytree",
                0.7,
                1.0
            ),

        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "random_state": RANDOM_STATE,
        "n_jobs": -1
    }

    model = XGBClassifier(**params)
    scores = cross_val_score(
        model,
        X,
        y,
        cv=3,
        scoring="accuracy",
        n_jobs=-1
    )

    return scores.mean()

def objective_cat(trial):
    params = {
        "iterations": 300,
        "learning_rate":
            trial.suggest_float(
                "learning_rate",
                0.01,
                0.1,
                log=True
            ),

        "depth":
            trial.suggest_int(
                "depth",
                4,
                10
            ),

        "verbose": 0,
        "random_seed": RANDOM_STATE
    }

    model = CatBoostClassifier(**params)
    scores = cross_val_score(
        model,
        X,
        y,
        cv=3,
        scoring="accuracy",
        n_jobs=-1
    )

    return scores.mean()

## Optuna

In [16]:
print("LIGHTGBM")
study_lgbm = optuna.create_study(
    direction="maximize"
)

study_lgbm.optimize(
    objective_lgbm,
    n_trials=N_TRIALS
)

print("\nBEST LGBM SCORE:")
print(study_lgbm.best_value)

print("\nBEST LGBM PARAMS:")
print(study_lgbm.best_params)

print("XGBOOST")
study_xgb = optuna.create_study(
    direction="maximize"
)

study_xgb.optimize(
    objective_xgb,
    n_trials=N_TRIALS
)

print("\nBEST XGB SCORE:")
print(study_xgb.best_value)

print("\nBEST XGB PARAMS:")
print(study_xgb.best_params)


print("CATBOOST")
study_cat = optuna.create_study(
    direction="maximize"
)

study_cat.optimize(
    objective_cat,
    n_trials=N_TRIALS
)

print("\nBEST CATBOOST SCORE:")
print(study_cat.best_value)

print("\nBEST CATBOOST PARAMS:")
print(study_cat.best_params)



[I 2026-05-21 00:17:40,800] A new study created in memory with name: no-name-99f3fe85-b7d4-4fce-bdca-c3692de4c94a


LIGHTGBM


[I 2026-05-21 00:29:58,616] Trial 0 finished with value: 0.7995066765302198 and parameters: {'learning_rate': 0.03927165688581448, 'max_depth': 6, 'num_leaves': 72, 'subsample': 0.7924106016061077, 'colsample_bytree': 0.9119900936165362}. Best is trial 0 with value: 0.7995066765302198.
[I 2026-05-21 00:34:58,431] Trial 1 finished with value: 0.7995070815710008 and parameters: {'learning_rate': 0.09633631867725846, 'max_depth': 4, 'num_leaves': 87, 'subsample': 0.7077433168124431, 'colsample_bytree': 0.8454026311890517}. Best is trial 1 with value: 0.7995070815710008.
[I 2026-05-21 00:52:00,127] Trial 2 finished with value: 0.8009860474792044 and parameters: {'learning_rate': 0.010780541786163632, 'max_depth': 6, 'num_leaves': 91, 'subsample': 0.7214409554056801, 'colsample_bytree': 0.9916725459438542}. Best is trial 2 with value: 0.8009860474792044.
[I 2026-05-21 01:08:31,996] Trial 3 finished with value: 0.7942477890443923 and parameters: {'learning_rate': 0.0880510490711576, 'max_dep


BEST LGBM SCORE:
0.8009860474792044

BEST LGBM PARAMS:
{'learning_rate': 0.010780541786163632, 'max_depth': 6, 'num_leaves': 91, 'subsample': 0.7214409554056801, 'colsample_bytree': 0.9916725459438542}
XGBOOST


[I 2026-05-21 01:14:07,521] Trial 0 finished with value: 0.7988494573587648 and parameters: {'learning_rate': 0.042917998449271265, 'max_depth': 8, 'subsample': 0.7689247969474738, 'colsample_bytree': 0.8828234366912375}. Best is trial 0 with value: 0.7988494573587648.
[I 2026-05-21 01:14:10,601] Trial 1 finished with value: 0.7965491497545939 and parameters: {'learning_rate': 0.06569114321774427, 'max_depth': 6, 'subsample': 0.8568225758652634, 'colsample_bytree': 0.8285523484342052}. Best is trial 0 with value: 0.7988494573587648.
[I 2026-05-21 01:14:13,828] Trial 2 finished with value: 0.793590569872937 and parameters: {'learning_rate': 0.08360939743725203, 'max_depth': 7, 'subsample': 0.9823622147318485, 'colsample_bytree': 0.8998907311903797}. Best is trial 0 with value: 0.7988494573587648.
[I 2026-05-21 01:14:14,920] Trial 3 finished with value: 0.8046004693936605 and parameters: {'learning_rate': 0.030347768304254764, 'max_depth': 6, 'subsample': 0.9642676344326093, 'colsample_b


BEST XGB SCORE:
0.8046004693936605

BEST XGB PARAMS:
{'learning_rate': 0.030347768304254764, 'max_depth': 6, 'subsample': 0.9642676344326093, 'colsample_bytree': 0.9414432851923668}
CATBOOST


[I 2026-05-21 01:14:18,255] Trial 0 finished with value: 0.8055880398262018 and parameters: {'learning_rate': 0.09928166473392641, 'depth': 6}. Best is trial 0 with value: 0.8055880398262018.
[I 2026-05-21 01:14:19,876] Trial 1 finished with value: 0.7930986883483376 and parameters: {'learning_rate': 0.013215389338195471, 'depth': 4}. Best is trial 0 with value: 0.8055880398262018.
[I 2026-05-21 01:14:27,601] Trial 2 finished with value: 0.8014794681587721 and parameters: {'learning_rate': 0.08341369877114291, 'depth': 9}. Best is trial 0 with value: 0.8055880398262018.
[I 2026-05-21 01:14:29,605] Trial 3 finished with value: 0.7980287637280471 and parameters: {'learning_rate': 0.014174539284833882, 'depth': 6}. Best is trial 0 with value: 0.8055880398262018.
[I 2026-05-21 01:14:31,551] Trial 4 finished with value: 0.8041097219832483 and parameters: {'learning_rate': 0.02074065400242557, 'depth': 6}. Best is trial 0 with value: 0.8055880398262018.



BEST CATBOOST SCORE:
0.8055880398262018

BEST CATBOOST PARAMS:
{'learning_rate': 0.09928166473392641, 'depth': 6}


## 7. Improved model

Try at least one stronger model and compare the result.


In [17]:

lgbm_model = LGBMClassifier(
    **study_lgbm.best_params,
    random_state=RANDOM_STATE,
    verbosity=-1
)

xgb_model = XGBClassifier(
    **study_xgb.best_params,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE
)

cat_model = CatBoostClassifier(
    **study_cat.best_params,
    loss_function="Logloss",
    verbose=0,
    random_seed=RANDOM_STATE
)

et_model = ExtraTreesClassifier(
    n_estimators=2000,
    max_depth=14,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# STACKING
improved_model = StackingClassifier(
    estimators=[
        ("cat", cat_model),
        ("lgbm", lgbm_model),
        ("xgb", xgb_model),
        ("et", et_model)
    ],

    final_estimator=LogisticRegression(
        max_iter=5000
    ),

    stack_method="predict_proba",
    cv=5,
    n_jobs=-1
)

oof_probs = cross_val_predict(
    improved_model,
    X,
    y,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

## 8. Compare models

Briefly discuss the difference between the baseline and the improved model.


In [18]:
best_threshold = 0.5
best_acc = 0

for threshold in np.arange(0.30, 0.71, 0.005):
    preds = (
        oof_probs > threshold
    ).astype(int)

    acc = accuracy_score(
        y,
        preds
    )

    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold

final_preds = (
    oof_probs > best_threshold
).astype(int)

final_acc = accuracy_score(
    y,
    final_preds
)

final_f1 = f1_score(
    y,
    final_preds
)

final_auc = roc_auc_score(
    y,
    oof_probs
)

print(f"BEST THRESHOLD : {best_threshold:.4f}")
print(f"OOF ACCURACY  : {final_acc:.6f}")
print(f"OOF F1 SCORE  : {final_f1:.6f}")
print(f"OOF ROC AUC   : {final_auc:.6f}")

BEST THRESHOLD : 0.4750
OOF ACCURACY  : 0.807067
OOF F1 SCORE  : 0.810950
OOF ROC AUC   : 0.893323


**Write a short discussion here.**

- Which model performed better?
- Was the improvement large or small?
- What might explain the difference?


## 9. Train the final model on the full student training set

Choose your final model and fit it using all available labeled data.


In [19]:
final_model = improved_model

final_model.fit(X, y)
test_probabilities = final_model.predict_proba(
    test_df
)[:, 1]

test_predictions = (
    test_probabilities > best_threshold
).astype(bool)

submission = pd.DataFrame({
    ID_COL: test_ids,
    "prediction": test_predictions
})

print(submission.head())

submission.to_csv(
    "submission.csv",
    index=False
)

print("\nsubmission.csv gerado co m sucesso.")



  PassengerId  prediction
0     8137_01       False
1     7191_01       False
2     6837_01        True
3     4453_01        True
4     4400_03        True

submission.csv gerado co m sucesso.


## 10. Save the submission file


In [20]:
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")


Saved submission.csv


## 11. Submit to the leaderboard

Upload `submission.csv` to:

<https://(Enter your Streamlit app URL here)/>

After submitting, record your score below.


**Leaderboard score(s):**

- First submission:
- Best submission:
- Final submitted model:


## 12. Final reflection

Write a short final reflection addressing:

- what preprocessing choices were most important
- whether feature engineering helped
- what model worked best for your group
- what you would try next if you had more time


**Write your final reflection here.**


## 📤 Final Deliverables Checklist

Before submitting your work, verify that you are delivering:

- [ ] completed notebook
- [ ] generated submission file
- [ ] leaderboard score recorded
- [ ] short discussion of preprocessing and model choices
- [ ] final reflection
